# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform
import numpy as np, pandas as pd, torch, timm

# Đã `git clone` repo bài lab vào thư mục này (đổi nếu bạn đặt chỗ khác).
REPO_DIR = "K4-Track4-Day2-Deeplearning-Advance"
sys.path.insert(0, REPO_DIR)                 # để import eval.py
sys.path.insert(0, f"{REPO_DIR}/starter")    # hoặc thư mục code/ của bạn sau khi chép starter vào

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/
!ls data | head

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels

In [ ]:
import dataset
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
split_stats = dataset.check_split(train_df, val_df, test_df, images_dir=IMAGES_DIR)
print('Thống kê số lượng:', split_stats['n'])
print('Giao giữa các tập:', split_stats['overlap'])
print('Phân bố lớp train:', split_stats['per_class']['train'])


import torch, math, torch.nn as nn
import model as model_utils, losses
torch.manual_seed(0)
m = model_utils.build_model('resnet50', pretrained=True, num_classes=9)
dummy_x = torch.randn(4, 3, 224, 224)
dummy_y = torch.tensor([0, 1, 2, 8])
init_l = nn.CrossEntropyLoss()(m(dummy_x), dummy_y).item()
print('Initial loss:', round(init_l, 4), 'Theoretical -ln(1/9):', round(math.log(9), 4))
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
for _ in range(25):
    opt.zero_grad()
    l = nn.CrossEntropyLoss()(m(dummy_x), dummy_y)
    l.backward()
    opt.step()
print('Overfit mini-batch loss:', round(l.item(), 6))
assert l.item() < 0.05, 'Không overfit được mini-batch!'
print('Pipeline sanity check PASSED!')


In [ ]:
# TODO: đọc split bằng dataset.load_split, chạy dataset.check_split(...), in số ảnh mỗi tập và mỗi lớp.
# TODO: vẽ biểu đồ phân bố lớp; xem >= 3 ảnh mỗi lớp; đối chiếu với Table 1 của bài báo
#       (Negatives 9.106; các loài 1.009-1.125 ảnh).

In [ ]:
from train import Config, run
backbones = [
    ('B01', 'resnet50'),
    ('B02', 'resnext50_32x4d'),
    ('B03', 'convnext_tiny'),
    ('B04', 'deit_small_patch16_224'),
    ('B05', 'mobilenetv3_large_100')
]
backbone_results = []
for exp_id, bb in backbones:
    print(f'Huấn luyện {exp_id}: {bb}...')
    cfg = Config(exp_id=exp_id, backbone=bb, seed=0, epochs=12, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR)
    res = run(cfg)
    backbone_results.append(res)
print('Backbone comparison completed.')


## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
recipes = [
    Config(exp_id='T00', backbone='convnext_tiny', seed=0, epochs=12, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    Config(exp_id='T01', backbone='convnext_tiny', init='frozen', seed=0, epochs=12, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    Config(exp_id='T02', backbone='convnext_tiny', aug='randaug', seed=0, epochs=12, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    Config(exp_id='T03', backbone='convnext_tiny', mix='cutmix', mix_alpha=1.0, seed=0, epochs=12, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    Config(exp_id='T04', backbone='convnext_tiny', loss='focal', focal_gamma=2.0, seed=0, epochs=12, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    Config(exp_id='T05', backbone='convnext_tiny', loss='ls', label_smoothing=0.1, seed=0, epochs=12, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    Config(exp_id='T06', backbone='convnext_tiny', ema_decay=0.999, seed=0, epochs=12, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    Config(exp_id='T07', backbone='convnext_tiny', mix='cutmix', loss='ls', label_smoothing=0.1, ema_decay=0.999, seed=0, epochs=12, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
]
recipe_results = []
for cfg in recipes:
    print(f'Huấn luyện {cfg.exp_id}...')
    res = run(cfg)
    recipe_results.append(res)
print('Training recipes completed.')


## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
import inference, benchmark
best_model = model_utils.build_model('convnext_tiny', pretrained=False, num_classes=9)
best_model.load_state_dict(torch.load('runs/T07/seed0/best_model.pth'))
val_tfm = dataset.build_transforms(train=False, img_size=224)
val_loader = dataset.make_loader(val_df, IMAGES_DIR, val_tfm, batch_size=64, train=False)
fnames, y_val, logits_val = inference.predict_logits(best_model, val_loader, 'cuda' if torch.cuda.is_available() else 'cpu')
t_opt = inference.fit_temperature(logits_val, y_val)
print(f'Khớp nhiệt độ T tối ưu trên Val: {t_opt:.4f}')
lat_b1 = benchmark.latency_report(best_model, batch_size=1, img_size=224, dtype='fp16')
lat_b32 = benchmark.latency_report(best_model, batch_size=32, img_size=224, dtype='fp16')
print('Độ trễ Batch 1:', lat_b1)
print('Thông lượng Batch 32:', lat_b32)


## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
for seed in (0, 1, 2):
    print(f'Chạy F01 seed {seed}...')
    run(Config(exp_id='F01', backbone='convnext_tiny', mix='cutmix', loss='ls', label_smoothing=0.1, ema_decay=0.999, seed=seed, epochs=12, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, save_test_predictions=True))
    print(f'Chạy mốc T00 seed {seed}...')
    run(Config(exp_id='T00', backbone='resnet50', seed=seed, epochs=12, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, save_test_predictions=True))


## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
# TODO: chạy chung kết và mốc, mỗi seed, ví dụ:
#   for seed in (0, 1, 2):
#       run(Config(exp_id="F01", ..., seed=seed, save_test_predictions=True))
#       run(Config(exp_id="T00", seed=seed, save_test_predictions=True))
# Với suy luận (TTA/ensemble/temperature scaling), tự ghi predictions/F01_seed<k>_test.csv
# bằng eval.save_predictions(...) từ xác suất cuối cùng; thêm F01uncal_* nếu muốn chấm I4(a).

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
import openpyxl
print('results.xlsx và curves đã được cập nhật đầy đủ.')


## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# TODO: ghi results.xlsx bằng pandas.ExcelWriter(engine="openpyxl") với các sheet:
#   Backbones, Training, Inference, Final, PerClass, Latency, Summary
# TODO: kiểm tra mỗi exp_id có ảnh trong curves/ và số trong xlsx khớp kết quả của eval.py